# 05 · MonkeyOCR

**MonkeyOCR** — модель с парадигмой **Structure → Recognition → Relation**. Чекпоинт: [`echo840/MonkeyOCR`](https://huggingface.co/echo840/MonkeyOCR).

Ноутбук берёт subset, сохранённый в `data/subset.json` ноутбуком `01_setup_and_dataset.ipynb`, и прогоняет на нём модель. Результаты записываются в `results/<model>/predictions.jsonl`.

## 1. Установка

In [ ]:
!pip install -q -r requirements.txt

## 2. Установка

In [ ]:
import os, sys, pathlib

# Клонируем только если ещё нет (защита от повторного запуска)
if not pathlib.Path('ocr_eval').exists():
    !git clone https://github.com/AStrateg2509/ocr_eval.git

os.chdir('ocr_eval')
sys.path.insert(0, 'src')
print("CWD =", os.getcwd())

## 3. Конфиг и модель

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord
cfg = load_config('configs/monkeyocr.yaml')
cfg

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

## 4. Subset

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

## 5. Инференс

MonkeyOCR имеет встроенный метод `model.chat_full_page(...)` (см. README модели). Если интерфейс изменится — заменить на актуальный.

In [ ]:
import traceback
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='monkeyocr')
        try:
            img = Image.open(img_path).convert('RGB')
            with Timer('infer') as t, torch.no_grad():
                if hasattr(model, 'chat_full_page'):
                    out = model.chat_full_page(tokenizer, img,
                                               max_new_tokens=cfg['inference']['max_new_tokens'])
                else:
                    out = model.chat(tokenizer, img,
                                     query='Convert this page to markdown',
                                     max_new_tokens=cfg['inference']['max_new_tokens'])
            rec.full_text = out if isinstance(out, str) else str(out)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

## 6. Освободить GPU

In [ ]:
del model; cuda_free(); print(gpu_info())